# Qiskit Metal Interactive Design Explorer
Use this notebook to visually explore and test Qiskit Metal designs before批量生成

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from qiskit_metal import designs, draw
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket
from qiskit_metal.qlibrary.couplers.coupled_line import CoupledLine
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.toolbox_python.load_design import load_design_from_json
import json
from pathlib import Path

%matplotlib inline

In [ ]:
# Create a new design
design = designs.DesignPlanar("Interactive_Design")
design.overwrite_enabled = True
design.add_ground_plane()
print("Design created successfully")

In [ ]:
# Define parameters (adjust these to explore different designs)
params = {
    'junction_width_nm': 200,
    'junction_length_nm': 150,
    'pad_area_um2': 100,
    'gap_to_ground_um': 20,
    'finger_length_um': 50,
    'finger_width_um': 5,
    'finger_count': 10,
    'finger_gap_um': 3,
    'hbar_thickness_um': 10,
    'beam_length_um': 200,
    'beam_width_um': 15,
}

print("Current parameters:")
for k, v in params.items():
    print(f"  {k}: {v}")

In [ ]:
# Create transmon qubit
transmon_options = dict(
    pos_x='0mm',
    pos_y='0mm',
    junction_width=f"{params['junction_width_nm']}nm",
    junction_length=f"{params['junction_length_nm']}nm",
    pad_width=f"{np.sqrt(params['pad_area_um2']*1e6):.0f}nm",
    pad_height=f"{np.sqrt(params['pad_area_um2']*1e6):.0f}nm",
    gap=f"{params['gap_to_ground_um']}um",
    connection_pads=dict(
        readout=dict(loc_W=1, pad_width='175um'),
        drive=dict(loc_W=0, pad_width='100um')
    )
)

q1 = TransmonPocket(design, 'Q1', options=transmon_options)
print("Transmon added")

In [ ]:
# Create coupling capacitor
cap_options = dict(
    pos_x='200um',
    pos_y='0um',
    finger_length=f"{params['finger_length_um']}um",
    finger_width=f"{params['finger_width_um']}um",
    finger_number=params['finger_count'],
    finger_gap=f"{params['finger_gap_um']}um",
)

cap = CoupledLine(design, 'C1', options=cap_options)
print("Capacitor added")

In [ ]:
# Connect components
design.connect_components('Q1', 'C1')
print("Components connected")

In [ ]:
# Plot the design
fig, ax = design.plot()
ax.set_title("Interactive Design Preview")
plt.show()

In [ ]:
# Check for overlaps
overlaps = design.check_overlaps()
if overlaps:
    print("⚠️ Overlaps detected:")
    print(overlaps)
else:
    print("✅ No overlaps - design is valid")

In [ ]:
# Export to JSON
json_str = design.save_to_string()
layout_data = json.loads(json_str)
print(f"Design exported, {len(json_str)} characters")

In [ ]:
# Save to file
output_dir = Path.home() / 'projects' / 'qsymphony' / 'phase1_hardware' / 'test_layouts'
output_dir.mkdir(exist_ok=True)

filename = output_dir / 'test_layout.json'
with open(filename, 'w') as f:
    json.dump(layout_data, f, indent=2)
    
print(f"Layout saved to: {filename}")